# Tokenization & subword algorithms: build it, prove it, see the trade-offs

A neural network can only multiply numbers, so before a transformer runs, a **tokenizer** turns a
string into a sequence of integers. This notebook builds the core subword algorithms **from
scratch** in pure Python and proves each one's headline claim with an `assert` *before* it prints —
so every number you see is checked, not asserted by prose.

We import the canonical functions from [`tokenization.py`](tokenization.py) (the same module the
concept page and every figure use), so nothing here can drift from the page. We will:

1. **Train BPE** on the classic toy corpus and read off the ordered merge list.
2. **Encode unseen words** by replaying the merges — the no-`[UNK]` coverage guarantee.
3. Show **BPE and WordPiece pick different first merges** on identical data.
4. **Viterbi-decode** the best segmentation under a Unigram model.
5. **Sweep vocabulary size** to expose the compression-vs-OOV trade-off.
6. Tokenize a real sentence with the **production GPT-4 and BERT tokenizers** and measure the
   multilingual tax.

> **Device:** `cpu (pure-Python/numpy)` — the subword algorithms are integer/string bookkeeping
> with no tensors to accelerate, so results are bit-for-bit identical on any machine.

## Setup: import the canonical module

Everything below comes from `tokenization.py`. Importing (rather than re-defining) is what
guarantees the notebook, the page, and the figures share one source of truth.

In [1]:
import sys

from tokenization import (
    TOY_CORPUS,
    N_MERGES,
    train_bpe,
    encode_bpe,
    bpe_base_alphabet,
    first_merge_bpe_vs_wordpiece,
    viterbi_segment,
    train_corpus_words,
    vocab_size_sweep,
)

print("device: cpu (pure-Python/numpy)")
print("python:", sys.version.split()[0])
try:
    import numpy
    print("numpy:", numpy.__version__)
except ImportError:
    print("numpy: (not required for the pure-Python algorithms)")
print("toy corpus:", TOY_CORPUS)

device: cpu (pure-Python/numpy)
python: 3.12.13
numpy: 2.4.6
toy corpus: {'low': 5, 'lower': 2, 'newest': 6, 'widest': 3}


## Step 1 — train BPE and read the ordered merge list

BPE starts from single characters and, at each step, merges the **most frequent adjacent pair**,
recording the merge. The saved artifact is the *ordered* list of merges — encoding will replay it
in exactly this order. The base alphabet has 11 symbols; each merge adds exactly one token.

The assert pins the first merge to `(e, s)` (frequency 9, from `newest`×6 + `widest`×3) — if the
algorithm ever broke, this cell would fail instead of quietly printing a wrong list.

In [2]:
merges, freqs, _ = train_bpe(TOY_CORPUS, N_MERGES)

assert merges[0] == ("e", "s"), "first merge must be (e, s) at frequency 9"
assert len(merges) == N_MERGES == 10
assert len(bpe_base_alphabet(TOY_CORPUS)) == 11  # l o w e r n s t i d </w>

print(f"base alphabet: {len(bpe_base_alphabet(TOY_CORPUS))} symbols")
print("ordered merge list (the saved artifact):")
for i, (pair, freq) in enumerate(zip(merges, freqs), 1):
    print(f"  step {i:2d}: merge {pair!s:>20}  (freq {freq})")

base alphabet: 11 symbols
ordered merge list (the saved artifact):
  step  1: merge           ('e', 's')  (freq 9)
  step  2: merge          ('es', 't')  (freq 9)
  step  3: merge      ('est', '</w>')  (freq 9)
  step  4: merge           ('l', 'o')  (freq 7)
  step  5: merge          ('lo', 'w')  (freq 7)
  step  6: merge           ('n', 'e')  (freq 6)
  step  7: merge          ('ne', 'w')  (freq 6)
  step  8: merge   ('new', 'est</w>')  (freq 6)
  step  9: merge      ('low', '</w>')  (freq 5)
  step 10: merge           ('w', 'i')  (freq 3)


## Step 2 — encode an *unseen* word: the no-`[UNK]` guarantee

Encoding replays the learned merges, in order, until none apply. The word `lowest` was **never in
the training corpus**, yet it encodes cleanly to `low` + `est</w>` — both learned pieces. And a word
made of rare letters (`qux`) still encodes: it just falls back to single characters, which are always
in the base vocabulary. **There is no `[UNK]`, ever** — that is the whole coverage promise.

In [3]:
seg = encode_bpe("lowest", merges)
assert seg == ["low", "est</w>"], f"expected ['low', 'est</w>'], got {seg}"

# worst case: a word of rare letters falls to single characters -- still fully covered
qux = encode_bpe("qux", merges)
assert all(len(p) == 1 for p in qux if p != "</w>"), "qux should fall to single chars"

for w in ("lowest", "newer", "qux"):
    print(f"encode {w!r:>10} -> {encode_bpe(w, merges)}")

encode   'lowest' -> ['low', 'est</w>']
encode    'newer' -> ['new', 'e', 'r', '</w>']
encode      'qux' -> ['q', 'u', 'x', '</w>']


## Step 3 — BPE vs WordPiece: different first merge on identical data

This is the cleanest way to show the two algorithms differ. On the **same** corpus:

- **BPE** ranks pairs by raw frequency → first merge is `(e, s)` (frequency 9).
- **WordPiece** ranks by `freq(a,b) / (freq(a)·freq(b))` → first merge is `(i, d)`.

`(i, d)` occurs only **3** times, far less than `(e, s)`'s 9 — but `i` and `d` *each* appear only 3
times and *always together* (in `widest`), so their normalized score `3 / (3·3) = 0.333` is highest.
WordPiece rewards pairs that are **frequent together but rare apart**, not just frequent.

In [4]:
bpe_pick, wp_pick, details = first_merge_bpe_vs_wordpiece(TOY_CORPUS)

assert bpe_pick == ("e", "s"), "BPE picks highest raw frequency"
assert wp_pick == ("i", "d"), "WordPiece picks highest normalized score"
assert bpe_pick != wp_pick, "the whole point: they disagree"

print(f"BPE first merge      : {bpe_pick}  (raw freq {int(details[bpe_pick]['pair_freq'])})")
print(f"WordPiece first merge: {wp_pick}  "
      f"(freq {int(details[wp_pick]['pair_freq'])}, score {details[wp_pick]['score']:.3f})")
print()
print(f"{'pair':>8} {'freq':>5} {'denom':>6} {'score':>7}")
for pair in [("e", "s"), ("s", "t"), ("l", "o"), ("i", "d")]:
    d = details[pair]
    print(f"{str(pair):>8} {int(d['pair_freq']):>5} {int(d['denom']):>6} {d['score']:>7.3f}")

BPE first merge      : ('e', 's')  (raw freq 9)
WordPiece first merge: ('i', 'd')  (freq 3, score 0.333)

    pair  freq  denom   score
('e', 's')     9    153   0.059
('s', 't')     9     81   0.111
('l', 'o')     7     49   0.143
('i', 'd')     3      9   0.333


## Step 4 — Unigram LM: Viterbi-decode the best segmentation

Unigram tokenization is **top-down and probabilistic**: each piece has a log-probability, a
segmentation's score is the **sum** of its pieces' log-probs, and decoding finds the highest-scoring
split with **Viterbi** (dynamic programming over split points) — a *global* optimum for this string,
not a greedy merge replay. Here `low` and `est` are cheap pieces, so `lowest` decodes to `low | est`.

In [5]:
import math

# a tiny hand-built unigram model; 'low' and 'est' are cheap, single chars are expensive
pieces = {"low": 0.30, "est": 0.30, "lo": 0.05, "we": 0.05,
          "l": 0.05, "o": 0.05, "w": 0.05, "e": 0.05, "s": 0.05, "t": 0.05}
logp = {p: math.log(v) for p, v in pieces.items()}

seg_u, score_u = viterbi_segment("lowest", logp)
assert seg_u == ["low", "est"], f"Viterbi should pick ['low', 'est'], got {seg_u}"

# verify it really is the best: it must beat the all-characters segmentation
chars_score = sum(logp[c] for c in "lowest")
assert score_u > chars_score, "the chosen split must score higher than splitting into characters"

print(f"Viterbi best segmentation of 'lowest': {seg_u}")
print(f"  its log-prob          : {score_u:.3f}")
print(f"  all-characters log-prob: {chars_score:.3f}  (worse, as it must be)")

Viterbi best segmentation of 'lowest': ['low', 'est']
  its log-prob          : -2.408
  all-characters log-prob: -17.974  (worse, as it must be)


## Step 5 — vocab-size sweep: compression vs OOV-floor

The one hyperparameter you set when training a tokenizer is **vocabulary size**. Sweeping the number
of merges (= vocabulary size) on a larger corpus shows the trade-off directly: as the vocabulary
grows, **tokens-per-word falls** (better compression, shorter sequences) *and* the share of held-out
words that fall all the way to single characters **falls too** (better coverage). Both gains are paid
for in embedding-table parameters and per-token data sparsity — which is why bigger isn't simply
better. This is exactly the `tok_vocab_sweep.png` figure on the page.

In [6]:
corpus = train_corpus_words()
held_out = ["lowness", "newness", "broadest", "renewing", "widening", "boldness"]
rows = vocab_size_sweep(corpus, held_out, merge_counts=[0, 2, 5, 10, 20, 40])

# both curves must be monotone-ish down: more merges -> better compression AND less fallback
assert rows[-1]["tokens_per_word"] < rows[0]["tokens_per_word"], "more merges -> shorter sequences"
assert rows[-1]["oov_fallback_rate"] < rows[0]["oov_fallback_rate"], "more merges -> less fallback"

print(f"{'merges':>6} {'vocab':>6} {'tok/word':>9} {'char-fallback':>14}")
for r in rows:
    print(f"{int(r['n_merges']):>6} {int(r['vocab_size']):>6} "
          f"{r['tokens_per_word']:>9.2f} {r['oov_fallback_rate']:>13.0%}")

merges  vocab  tok/word  char-fallback
     0     15      4.31          100%
     2     17      3.96          100%
     5     20      3.47           83%
    10     25      2.84           67%
    20     35      1.91           17%
    40     55      1.14            0%


## Step 6 — real production tokenizers: GPT-4 vs BERT, and the multilingual tax

Theory done — now the real thing. We tokenize one sentence with **GPT-4's `cl100k_base`** (byte-level
BPE, via `tiktoken`) and **BERT's `bert-base-uncased`** (WordPiece, via `transformers`), then measure
how many tokens the *same meaning* costs across four languages. These are the exact numbers in the
`tok_gpt4_vs_bert.png` and `tok_multilingual_cost.png` figures.

> Requires `tiktoken` and `transformers`. If they aren't installed, this cell prints a note and the
> from-scratch results above still stand on their own.

In [7]:
try:
    from tokenization import gpt4_tokens, bert_tokens, multilingual_gpt4_counts

    gpt = gpt4_tokens()
    bert = bert_tokens()
    assert len(gpt) == 16, f"GPT-4 should give 16 tokens, got {len(gpt)}"
    assert len(bert) == 17, f"BERT should give 17 tokens, got {len(bert)}"

    print(f"GPT-4 ({len(gpt):2d} tokens): {gpt}")
    print(f"BERT  ({len(bert):2d} tokens): {bert}")
    print()

    counts = multilingual_gpt4_counts()
    assert dict(counts)["English"] == 7 and dict(counts)["Hindi"] == 27
    base = counts[0][1]
    print("multilingual tax (GPT-4 tokens for the same sentence):")
    for lang, n in counts:
        print(f"  {lang:>8}: {n:2d} tokens  ({n/base:.1f}x English)")
except ImportError as exc:
    print(f"(skipped real-tokenizer demo: {exc})")
    print("the from-scratch results above do not depend on it.")

GPT-4 (16 tokens): ['The', ' unh', 'app', 'iest', ' developers', ' ref', 'act', 'ored', ' ', '123', '456', '7', ' lines', ' of', ' code', '.']
BERT  (17 tokens): ['the', 'un', '##ha', '##pp', '##iest', 'developers', 'ref', '##act', '##ored', '123', '##45', '##6', '##7', 'lines', 'of', 'code', '.']

multilingual tax (GPT-4 tokens for the same sentence):
   English:  7 tokens  (1.0x English)
   Spanish: 12 tokens  (1.7x English)
   Chinese: 12 tokens  (1.7x English)
     Hindi: 27 tokens  (3.9x English)


## What we built

- **BPE training** produced an *ordered merge list* — the saved artifact — by greedily merging the
  most-frequent adjacent pair (first merge `(e, s)`).
- **Encoding replayed those merges** to cover *unseen* words (`lowest` → `low` + `est</w>`), falling
  to single characters in the worst case — **never an `[UNK]`**.
- **BPE and WordPiece picked different first merges** on identical data (`(e,s)` vs `(i,d)`) — raw
  frequency vs frequency-normalized-by-association.
- **Unigram Viterbi** found the globally-best segmentation by dynamic programming, not greedy replay.
- The **vocab-size sweep** showed bigger vocabularies buy shorter sequences *and* better coverage —
  paid for in parameters and data sparsity.
- **Real GPT-4 and BERT tokenizers** confirmed the byte-level-BPE-vs-WordPiece differences and the
  2–4× multilingual tax — with the exact counts the figures use.

Every claim was checked by an `assert` before it printed. For the full narrative, see the
[concept page](../02-Tokenization-and-Subword-Algorithms.md); for the figures, see
[`make_figures_02.py`](make_figures_02.py).